# Labrador Minimal Downstream Ablation

This notebook is a compact downstream example for LabradorModel.

It does four things:
1. generates structured synthetic lab data,
2. creates binary labels from a nonlinear downstream rule,
3. trains LabradorModel as a classifier,
4. compares a small embedding-dimension sweep and a small learning-rate sweep.

## 1) Setup

This section imports the required libraries, defines a few constants, and sets the random seed so the example is reproducible.

In [21]:
import random

import numpy as np
import pandas as pd
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.models import LabradorModel


NUM_LABS = 10
LAB_CODES = [f"lab-{i}" for i in range(NUM_LABS)]
SEED = 42
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
DOWNSTREAM_SAMPLES = 600
EPOCHS = 5
BATCH_SIZE = 64


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.7.1+cu126
CUDA available: True


## 2) Synthetic Data

This section defines the synthetic downstream task used in the ablation.

- Samples are generated from latent clinical profiles with different lab co-occurrence patterns.
- Lab values depend on both code identity and profile-specific shifts.
- The downstream label follows the notebook rule: the target is positive when the interaction between `lab-2` and `lab-7` is large enough and `lab-1` stays below a threshold.

This produces a small nonlinear classification problem that is still fast to run.

In [22]:
def generate_structured_fake_batch(
    batch_size: int = 32,
    num_labs: int = NUM_LABS,
):
    """Generate synthetic lab bags with profile-specific code/value structure."""
    panel_probs = torch.tensor(
        [
            [0.05, 0.18, 0.07, 0.06, 0.05, 0.05, 0.18, 0.18, 0.10, 0.08],
            [0.12, 0.18, 0.16, 0.15, 0.06, 0.05, 0.13, 0.05, 0.05, 0.05],
            [0.10, 0.08, 0.05, 0.05, 0.16, 0.16, 0.10, 0.08, 0.12, 0.10],
        ],
        dtype=torch.float32,
    )

    anchor_codes = torch.tensor([1, 2, 7], dtype=torch.long)
    panel_ids = torch.randint(0, 3, (batch_size,))

    codes = torch.empty(batch_size, num_labs, dtype=torch.long)
    values = torch.empty(batch_size, num_labs, dtype=torch.float32)

    for i in range(batch_size):
        panel = panel_ids[i].item()
        extra_codes = torch.multinomial(
            panel_probs[panel],
            num_samples=num_labs - len(anchor_codes),
            replacement=True,
        )
        sample_codes = torch.cat([anchor_codes, extra_codes], dim=0)
        sample_codes = sample_codes[torch.randperm(num_labs)]

        sample_values = sample_codes.float() / (num_labs - 1)

        if panel == 0:
            sample_values[(sample_codes == 1) | (sample_codes == 6) | (sample_codes == 7)] += 0.18
            sample_values[sample_codes == 0] -= 0.05
        elif panel == 1:
            sample_values[(sample_codes == 2) | (sample_codes == 3)] += 0.15
            sample_values[sample_codes == 0] += 0.25
            sample_values[sample_codes == 1] += 0.05
            sample_values[sample_codes == 6] += 0.05
        else:
            sample_values[
                (sample_codes == 4)
                | (sample_codes == 5)
                | (sample_codes == 8)
                | (sample_codes == 9)
            ] += 0.18
            sample_values[sample_codes == 1] -= 0.06
            sample_values[sample_codes == 0] += 0.08

        sample_values += 0.05 * torch.randn_like(sample_values)
        sample_values = torch.clamp(sample_values, 0.0, 1.0)

        codes[i] = sample_codes
        values[i] = sample_values

    return codes, values


def get_mean_value_for_code(
    codes: torch.Tensor,
    values: torch.Tensor,
    target_code: int,
) -> torch.Tensor:
    mask = (codes == target_code).float()
    counts = mask.sum(dim=1)
    summed = (values * mask).sum(dim=1)
    return torch.where(
        counts > 0,
        summed / counts.clamp(min=1.0),
        torch.zeros_like(summed),
    )


def generate_downstream_labels(
    codes: torch.Tensor,
    values: torch.Tensor,
) -> torch.Tensor:
    code_1_value = get_mean_value_for_code(codes, values, target_code=1)
    code_2_value = get_mean_value_for_code(codes, values, target_code=2)
    code_7_value = get_mean_value_for_code(codes, values, target_code=7)
    return ((code_2_value * code_7_value > 0.20) & (code_1_value < 0.50)).long()


def make_downstream_samples(
    n_samples: int,
    seed: int,
    num_labs: int = NUM_LABS,
):
    set_seed(seed)
    codes, values = generate_structured_fake_batch(
        batch_size=n_samples,
        num_labs=num_labs,
    )
    labels = generate_downstream_labels(codes, values)

    samples = []
    for i in range(n_samples):
        samples.append(
            {
                "patient_id": f"patient-{i}",
                "visit_id": f"visit-{i}",
                "lab_codes": [LAB_CODES[int(code)] for code in codes[i].tolist()],
                "lab_values": values[i].tolist(),
                "label": int(labels[i].item()),
            }
        )
    return samples


def build_dataset(samples):
    return create_sample_dataset(
        samples=samples,
        input_schema={"lab_codes": "sequence", "lab_values": "tensor"},
        output_schema={"label": "binary"},
        dataset_name="labrador_downstream_demo",
    )


def split_samples(samples):
    n_samples = len(samples)
    train_end = int(n_samples * TRAIN_RATIO)
    val_end = int(n_samples * (TRAIN_RATIO + VAL_RATIO))
    return samples[:train_end], samples[train_end:val_end], samples[val_end:]

## 3) Training and Evaluation Helpers

This section defines the minimal training loop, evaluation function, and one helper that runs a single Labrador configuration on the downstream task.

In [23]:
def train_model(model, dataset, lr: float):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = get_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    for _ in range(EPOCHS):
        for batch in loader:
            optimizer.zero_grad()
            output = model(**batch)
            output["loss"].backward()
            optimizer.step()


def compute_binary_metrics(y_true: torch.Tensor, y_pred: torch.Tensor):
    y_true = y_true.float()
    y_pred = y_pred.float()

    tp = ((y_pred == 1) & (y_true == 1)).sum().item()
    tn = ((y_pred == 0) & (y_true == 0)).sum().item()
    fp = ((y_pred == 1) & (y_true == 0)).sum().item()
    fn = ((y_pred == 0) & (y_true == 1)).sum().item()

    total = max(tp + tn + fp + fn, 1)
    accuracy = (tp + tn) / total

    precision_den = max(tp + fp, 1)
    recall_den = max(tp + fn, 1)
    precision = tp / precision_den
    recall = tp / recall_den

    f1_den = max(precision + recall, 1e-12)
    f1 = 2 * precision * recall / f1_den

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


def evaluate_model(model, dataset):
    model.eval()
    loader = get_dataloader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for batch in loader:
            output = model(**batch)
            y_true = output["y_true"].detach().cpu().view(-1).float()
            y_prob = output["y_prob"].detach().cpu()

            if y_prob.dim() == 2 and y_prob.shape[1] > 1:
                y_pred = torch.argmax(y_prob, dim=1).view(-1).float()
            else:
                y_pred = (y_prob.view(-1) > 0.5).float()

            y_true_all.append(y_true)
            y_pred_all.append(y_pred)

    y_true_all = torch.cat(y_true_all)
    y_pred_all = torch.cat(y_pred_all)
    return compute_binary_metrics(y_true_all, y_pred_all)


def run_experiment(train_dataset, test_dataset, embed_dim: int, lr: float):
    set_seed(SEED)
    model = LabradorModel(
        dataset=train_dataset,
        code_feature_key="lab_codes",
        value_feature_key="lab_values",
        embed_dim=embed_dim,
        num_heads=2,
        num_layers=1,
    )
    train_model(model, train_dataset, lr=lr)
    return evaluate_model(model, test_dataset)

## 4) Create the Synthetic Downstream Dataset

This section generates one small synthetic dataset, shuffles it, splits it into train/validation/test subsets, and converts each split into a PyHealth dataset.

In [24]:
samples = make_downstream_samples(DOWNSTREAM_SAMPLES, seed=SEED)
random.Random(SEED).shuffle(samples)
train_samples, val_samples, test_samples = split_samples(samples)

train_dataset = build_dataset(train_samples)
val_dataset = build_dataset(val_samples)
test_dataset = build_dataset(test_samples)

print(f"train/val/test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"train positive rate: {np.mean([sample['label'] for sample in train_samples]):.3f}")
print(f"test positive rate: {np.mean([sample['label'] for sample in test_samples]):.3f}")

Label label vocab: {0: 0, 1: 1}
Label label vocab: {0: 0, 1: 1}
Label label vocab: {0: 0, 1: 1}
train/val/test: 360/120/120
train positive rate: 0.603
test positive rate: 0.600


## 5) Lightweight Masked-Lab Demonstration

This section demonstrates masking using the same `LabradorModel` encoder already used in this notebook.

It covers:
- masking exactly one lab position per sample,
- predicting masked lab code and masked lab value,
- optimizing with cross-entropy (code) + MSE (value),
- reporting evaluation metrics (accuracy, precision, recall, F1) for masked code prediction and MSE for masked value prediction.

Implementation note: we reuse `LabradorModel`'s embedding + transformer blocks and add lightweight prediction heads in-notebook only for this masked demo.

In [32]:
import torch.nn as nn
import torch.nn.functional as F


def mask_one_lab(codes: torch.Tensor, values: torch.Tensor, mask_token_id: int):
    """Mask one random lab position per sample."""
    batch_size, seq_len = codes.shape
    mask_idx = torch.randint(0, seq_len, (batch_size,))

    row_idx = torch.arange(batch_size)
    target_code = codes[row_idx, mask_idx].clone()
    target_value = values[row_idx, mask_idx].clone()

    masked_codes = codes.clone()
    masked_values = values.clone()

    masked_codes[row_idx, mask_idx] = mask_token_id
    masked_values[row_idx, mask_idx] = 0.0

    return masked_codes, masked_values, mask_idx, target_code, target_value


def encode_with_labrador(model: LabradorModel, codes: torch.Tensor, values: torch.Tensor):
    """Reuse LabradorModel encoder blocks to produce token representations."""
    code_emb = model.code_embedding(codes)
    value_emb = model.value_projection(values.unsqueeze(-1))

    x = code_emb + value_emb
    x = model.value_fusion(x)
    x = model.value_act(x)
    x = model.value_norm(x)

    # Keep all token positions active in this synthetic masked-lab demo.
    token_mask = torch.ones_like(codes, dtype=torch.bool)
    x = model.transformer(x, src_key_padding_mask=~token_mask)
    return x


def compute_multiclass_metrics(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    num_classes: int,
):
    """Macro precision/recall/F1 for masked code prediction."""
    y_true = y_true.long()
    y_pred = y_pred.long()

    acc = (y_true == y_pred).float().mean().item()

    precisions = []
    recalls = []
    f1s = []

    for cls in range(num_classes):
        tp = ((y_pred == cls) & (y_true == cls)).sum().item()
        fp = ((y_pred == cls) & (y_true != cls)).sum().item()
        fn = ((y_pred != cls) & (y_true == cls)).sum().item()

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall) / (precision + recall)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    return {
        "accuracy": float(acc),
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }

In [33]:
set_seed(SEED)

# Train/eval synthetic batches for masked prediction demo.
train_codes, train_values = generate_structured_fake_batch(batch_size=256, num_labs=NUM_LABS)
eval_codes, eval_values = generate_structured_fake_batch(batch_size=128, num_labs=NUM_LABS)

# Reuse the same LabradorModel encoder blocks.
masked_model = LabradorModel(
    dataset=train_dataset,
    code_feature_key="lab_codes",
    value_feature_key="lab_values",
    embed_dim=64,
    num_heads=2,
    num_layers=1,
)

# Introduce a dedicated mask token outside the normal code range [0, num_labs-1].
mask_token_id = masked_model.num_labs

# Expand embedding table by one row so mask token is valid.
old_embedding = masked_model.code_embedding
expanded_embedding = nn.Embedding(masked_model.num_labs + 1, masked_model.embed_dim)
with torch.no_grad():
    expanded_embedding.weight[: masked_model.num_labs].copy_(old_embedding.weight)
    expanded_embedding.weight[mask_token_id].copy_(old_embedding.weight.mean(dim=0))
masked_model.code_embedding = expanded_embedding

# Lightweight masked heads (not added to core model class).
mask_code_head = nn.Linear(masked_model.embed_dim, masked_model.num_labs)
mask_value_head = nn.Sequential(
    nn.Linear(masked_model.embed_dim + masked_model.num_labs, masked_model.embed_dim),
    nn.ReLU(),
    nn.Linear(masked_model.embed_dim, 1),
    nn.Sigmoid(),
)

optimizer = torch.optim.Adam(
    list(masked_model.parameters()) + list(mask_code_head.parameters()) + list(mask_value_head.parameters()),
    lr=1e-3,
)
ce_loss_fn = nn.CrossEntropyLoss()
mse_loss_fn = nn.MSELoss()

MASK_STEPS = 200
for _ in range(MASK_STEPS):
    masked_codes, masked_values, mask_idx, target_code, target_value = mask_one_lab(
        train_codes,
        train_values,
        mask_token_id,
    )

    x = encode_with_labrador(masked_model, masked_codes, masked_values)

    code_logits = mask_code_head(x)
    code_probs = torch.softmax(code_logits, dim=-1)
    value_pred = mask_value_head(torch.cat([x, code_probs], dim=-1)).squeeze(-1)

    row_idx = torch.arange(masked_codes.size(0))
    masked_code_logits = code_logits[row_idx, mask_idx]
    masked_value_pred = value_pred[row_idx, mask_idx]

    loss_code = ce_loss_fn(masked_code_logits, target_code)
    loss_value = mse_loss_fn(masked_value_pred, target_value)
    loss = loss_code + loss_value

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Evaluation on held-out synthetic batch
masked_codes, masked_values, mask_idx, target_code, target_value = mask_one_lab(
    eval_codes,
    eval_values,
    mask_token_id,
)

with torch.no_grad():
    x = encode_with_labrador(masked_model, masked_codes, masked_values)
    code_logits = mask_code_head(x)
    code_probs = torch.softmax(code_logits, dim=-1)
    value_pred = mask_value_head(torch.cat([x, code_probs], dim=-1)).squeeze(-1)

row_idx = torch.arange(masked_codes.size(0))
masked_code_logits = code_logits[row_idx, mask_idx]
masked_value_pred = value_pred[row_idx, mask_idx]

loss_code_eval = ce_loss_fn(masked_code_logits, target_code)
loss_value_eval = mse_loss_fn(masked_value_pred, target_value)
loss_total_eval = loss_code_eval + loss_value_eval

pred_code = torch.argmax(masked_code_logits, dim=1)
code_metrics = compute_multiclass_metrics(target_code, pred_code, num_classes=masked_model.num_labs)

masked_demo_results = pd.DataFrame([
    {
        "ce_loss": float(loss_code_eval.item()),
        "mse_loss": float(loss_value_eval.item()),
        "total_loss": float(loss_total_eval.item()),
        "accuracy": code_metrics["accuracy"],
        "precision": code_metrics["precision"],
        "recall": code_metrics["recall"],
        "f1": code_metrics["f1"],
    }
])

masked_demo_results

,ce_loss,mse_loss,total_loss,accuracy,precision,recall,f1
0,1.942994,0.089727,2.032721,0.3125,0.115682,0.133013,0.104244


## 6) Embedding Dimension Ablation

This section compares hidden sizes while keeping the learning rate fixed at `1e-3`.

In [27]:
embed_dim_results = []
for embed_dim in [64, 128, 256]:
    metrics = run_experiment(
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        embed_dim=embed_dim,
        lr=1e-3,
    )
    embed_dim_results.append({"embed_dim": embed_dim, **metrics})

embed_dim_df = pd.DataFrame(embed_dim_results)
embed_dim_df

,embed_dim,accuracy,precision,recall,f1
0,64,0.466667,0.590909,0.361111,0.448276
1,128,0.500000,0.603448,0.486111,0.538462
2,256,0.425000,0.548387,0.236111,0.330097


## 7) Learning Rate Ablation

This section compares learning rates while keeping the model size fixed at `embed_dim=128`.

In [28]:
learning_rate_results = []
for lr in [1e-4, 1e-3, 5e-3]:
    metrics = run_experiment(
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        embed_dim=128,
        lr=lr,
    )
    learning_rate_results.append({"learning_rate": lr, **metrics})

learning_rate_df = pd.DataFrame(learning_rate_results)
learning_rate_df

,learning_rate,accuracy,precision,recall,f1
0,0.0001,0.608333,0.610619,0.958333,0.745946
1,0.0010,0.500000,0.603448,0.486111,0.538462
2,0.0050,0.425000,0.551724,0.222222,0.316832


## 8) Summary

This section prints the final metric comparisons in a compact format, including accuracy, precision, recall, and F1.

In [29]:
print("Embedding Dimension Ablation")
for row in embed_dim_results:
    print(
        f"embed_dim={row['embed_dim']} "
        f"-> accuracy={row['accuracy']:.4f}, precision={row['precision']:.4f}, "
        f"recall={row['recall']:.4f}, f1={row['f1']:.4f}"
    )
print()

print("Learning Rate Ablation")
for row in learning_rate_results:
    print(
        f"lr={row['learning_rate']:.0e} "
        f"-> accuracy={row['accuracy']:.4f}, precision={row['precision']:.4f}, "
        f"recall={row['recall']:.4f}, f1={row['f1']:.4f}"
    )
print()

print("Embedding Dimension Results")
print(embed_dim_df.to_string(index=False))
print()

print("Learning Rate Results")
print(learning_rate_df.to_string(index=False))

Embedding Dimension Ablation
embed_dim=64 -> accuracy=0.4667, precision=0.5909, recall=0.3611, f1=0.4483
embed_dim=128 -> accuracy=0.5000, precision=0.6034, recall=0.4861, f1=0.5385
embed_dim=256 -> accuracy=0.4250, precision=0.5484, recall=0.2361, f1=0.3301

Learning Rate Ablation
lr=1e-04 -> accuracy=0.6083, precision=0.6106, recall=0.9583, f1=0.7459
lr=1e-03 -> accuracy=0.5000, precision=0.6034, recall=0.4861, f1=0.5385
lr=5e-03 -> accuracy=0.4250, precision=0.5517, recall=0.2222, f1=0.3168

Embedding Dimension Results
 embed_dim  accuracy  precision   recall       f1
        64  0.466667   0.590909 0.361111 0.448276
       128  0.500000   0.603448 0.486111 0.538462
       256  0.425000   0.548387 0.236111 0.330097

Learning Rate Results
 learning_rate  accuracy  precision   recall       f1
        0.0001  0.608333   0.610619 0.958333 0.745946
        0.0010  0.500000   0.603448 0.486111 0.538462
        0.0050  0.425000   0.551724 0.222222 0.316832
